In [ ]:
#import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
import os


In [ ]:
#load the dataset
labels_df = pd.read_csv('/content/drive/MyDrive/Ngao Project Folder/train_data.csv')

In [ ]:
#EDA
print(f"Number of images: {len(labels_df)}")
print(f"Dataset shape: {labels_df.shape}")
print(f"Columns: {labels_df.columns.tolist()}")
print()
print(labels_df.head(10))
print()
print("Missing values:")
print(labels_df.isnull().sum())

In [ ]:
#Class distribution
##calculating class weights based on the class frequencies
class_counts = labels_df['label'].value_counts()
total_samples = np.sum(class_counts)
print("Class Distribution:")
print(class_counts)
print()

In [ ]:
#due to imbalance in the data set we make the dataset balanced to reduce bias
#To handle the imbalance we balance the dataset using the min count (560) of all the classes.
class_counts = labels_df['label'].value_counts()
class_labels = labels_df['label'].unique() # Define class_labels

balanced_data = []
for class_label in class_labels:
    subset = labels_df[labels_df['label'] == class_label]
    num_images = min(560, len(subset))

    balanced_subset = subset.sample(n=num_images, random_state=42)

    balanced_data.append(balanced_subset)

balanced_df = pd.concat(balanced_data)

balanced_df.reset_index(drop=True, inplace=True)
balanced_df.head(300)

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image

num_samples = 4
train_dir = "/content/drive/MyDrive/Ngao Project Folder/Train"
disease_label = 'Salmonella'

print(f"Attempting to display images for: {disease_label}")

class_images = balanced_df[balanced_df["label"] == disease_label]["images"].sample(
    num_samples, random_state=42
)

plt.figure(figsize=(10, 8))

for i, image_file_name in enumerate(class_images):
    image_path = os.path.join(train_dir, image_file_name)
    try:
        image = Image.open(image_path)
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(image)
        plt.axis("off")
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
    except Exception as e:
        print(f"Error loading {image_file_name}: {e}")
        # If an error occurs, you might want to display a blank or error image instead

plt.suptitle(f"Sample Images - Disease {disease_label}")
plt.tight_layout() # Added for better layout
plt.show()

In [ ]:
#visualizing the balanced data set
sns.countplot(data=balanced_df, x="label")
plt.xlabel("Disease Label")
plt.ylabel("Count")
plt.title("Distribution of Disease Classes")
plt.show()

In [ ]:
balanced_df['label'].value_counts()

Iteration through each unique disease label in your balanced dataset (balanced_df). For each disease, it samples 4 image file names, constructs the full path to these images using the train_dir, opens each image using the PIL library, and then displays them in a single row using matplotlib. This helps in visually inspecting the images associated with each disease category after balancing the dataset. The code effectively creates a grid of sample images for every disease class, making sure to show num_samples (which is 4 in this case) images per class.

In [ ]:
num_samples = 4
train_dir = "/content/drive/MyDrive/Ngao Project Folder/Train"
for disease_label in class_labels:
    # Correctly filter balanced_df using its own label column
    class_images = balanced_df[balanced_df["label"] == disease_label]["images"].sample(
        num_samples, random_state=42 # Added random_state for reproducibility
    )
    plt.figure(figsize=(10, 8))

    for i, image_file_name in enumerate(class_images):
        # Construct path assuming images are directly in train_dir
        image_path = os.path.join(train_dir, image_file_name)
        image = Image.open(image_path)

        plt.subplot(1, num_samples, i + 1)
        plt.imshow(image)
        plt.axis("off")
    print(f"Displaying images for: {disease_label}") # Added print statement for clarity
    plt.suptitle(f"Sample Images - Disease {disease_label}")
    plt.tight_layout() # Added for better layout
    plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

# Define image dimensions and batch size
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Create an ImageDataGenerator for augmentation and preprocessing
# We'll use common augmentation techniques like rotation, shifts, shear, zoom, and horizontal flips.
data_gen = ImageDataGenerator(
    rescale=1./255, # Normalize pixel values to [0, 1]
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)


 #Splitting the data
train_df, test_df = train_test_split(balanced_df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)



# Note: This is just the definition of the ImageDataGenerator.
# To actually apply augmentation and create the augmented dataset,
# we would typically use flow_from_dataframe or flow_from_directory with this generator.
print("ImageDataGenerator for data augmentation has been configured.")
print("Next, we will apply this generator to create augmented training data.")

In [ ]:
train_generator = data_gen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,  # Path to the directory containing the images
    x_col='images',       # Column in dataframe containing image filenames
    y_col='label',        # Column in dataframe containing image labels
    target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
    batch_size=BATCH_SIZE,
    class_mode='categorical', # For one-hot encoded labels
    seed=42
)

# Get the class names and their indices
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"Found {train_generator.samples} training images belonging to {num_classes} classes: {class_names}.")
print("Augmented training data generator created successfully.")

In [ ]:
validation_generator = data_gen.flow_from_dataframe(
    dataframe=val_df,
    directory=train_dir,
    x_col='images',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

test_generator = data_gen.flow_from_dataframe(
    dataframe=test_df,
    directory=train_dir,
    x_col='images',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False, # Do not shuffle test data to maintain order for evaluation
    seed=42
)

print(f"Found {validation_generator.samples} validation images belonging to {num_classes} classes.")
print(f"Found {test_generator.samples} test images belonging to {num_classes} classes.")
print("Validation and test data generators created successfully.")

### Building the CNN Model

We will now define a Convolutional Neural Network (CNN) architecture using TensorFlow and Keras. This model will consist of several convolutional layers, pooling layers, and dense layers for classification. Dropout layers will be included to help prevent overfitting.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the CNN model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax') # Output layer with num_classes and softmax activation
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
print("CNN model defined and compiled successfully.")

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=30, # You can adjust the number of epochs
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

print("Model training complete.")

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#fit the model on test data
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")



In [ ]:
from sklearn.metrics import confusion_matrix


# Get true labels
y_true = test_generator.classes

# Get predictions
# Reset the test generator to ensure predictions start from the beginning
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)

# Get class labels from the generator
class_names = list(test_generator.class_indices.keys())

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

Coccidiosis (Index 0): The model correctly identified 78 out of 94 Coccidiosis cases. It misclassified 4 Coccidiosis cases as Healthy and 9 as New Castle Disease and 3 as salmonella. This class seems to be relatively well-identified. 80% correctly classified

Healthy (Index 1): Out of 109 true Healthy cases, 77 were correctly classified. However, a significant number were misclassified: 73as Coccidiosis, 20 as New Castle Disease, and 9 as Salmonella. This indicates the model struggles to accurately distinguish Healthy images, particularly from New Castle Disease. 70% correctly classified

New Castle Disease (Index 2): For 113 true New Castle Disease cases, 76 were correctly identified. Misclassifications include 9 as Healthy, and 3 as Salmonella. The confusion with 'Healthy' and 'Salmonella' is notable here. 67& correctly classified

Salmonella (Index 3): Out of 132 true Salmonella cases, 115 were correctly classified. A considerable portion was misclassified as Healthy 7, 2 as Coccidiosis and New Castle Disease 8. 87% correctly classifed.

Overall Observations:

The diagonal values (correct classifications) are generally the highest for each class, which is good.
There's significant confusion between Healthy, New Castle Disease, and Coccidiosis. These three classes seem to share features that make them harder for the model to distinguish from one another.
New Castle appears to be the most distinct class, with fewer misclassifications into other categories compared to the other diseases

In [ ]:
from sklearn.metrics import classification_report

# Generate the classification report
report = classification_report(y_true, y_pred, target_names=class_names)

print("Classification Report:")
print(report)

To load the model back in a new session or script, you would use `tf.keras.models.load_model()`:

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Get the unique classes and their counts from the training data
# We use train_df because weights should reflect the training distribution
class_labels = np.unique(train_df['label'])

# Calculate class weights
# 'balanced' mode automatically adjusts weights inversely proportional to class frequencies
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=class_labels,
    y=train_df['label']
)

# Create a dictionary mapping class indices to their weights
# The order of class_labels needs to match the order of class_indices from train_generator
class_weights = dict(zip([train_generator.class_indices[label] for label in class_labels], weights))

print("Calculated Class Weights:")
for class_idx, weight_val in class_weights.items():
    print(f"Class {class_idx} ({class_names[class_idx]}): {weight_val:.3f}")

Now, we can pass this `class_weights` dictionary to the `model.fit()` method. This will adjust the loss contribution of each sample during training based on its class.

First, we need to calculate the class weights. We'll use the `compute_class_weight` function from `sklearn.utils` and the labels from our training dataframe (`train_df`).

In [ ]:
# Re-train the model with class weights. You might want to save the original model first.
# For demonstration, we'll re-fit the existing model. If you want to compare,
# you should re-initialize the model or load its initial weights before fitting.

# Note: This will continue training from the last state if the model was already fitted.
# For a fresh start with weights, you would define and compile the model again.

print("Training model with class weights...")
history_weighted = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=30, # You can adjust the number of epochs
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    class_weight=class_weights # <--- This is where you apply the class weights
)

print("Model training with class weights complete.")

In [ ]:
#fit the model on test data
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
from sklearn.metrics import classification_report

# Generate the classification report
report = classification_report(y_true, y_pred, target_names=class_names)

print("Classification Report:")
print(report)

In [ ]:
from sklearn.metrics import confusion_matrix


# Get true labels
y_true = test_generator.classes

# Get predictions
# Reset the test generator to ensure predictions start from the beginning
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)

# Get class labels from the generator
class_names = list(test_generator.class_indices.keys())

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

Coccidiosis (True Label, Row 0):

Correctly classified: 82 out of 94 instances. This is an improvement from 78 previously. The model is quite effective here.
Misclassified: It misclassified 5 as Healthy (slight increase from 4), 4 as New Castle Disease (a significant decrease from 9, indicating better distinction), and 3 as Salmonella (same as before).
Healthy (True Label, Row 1):

Correctly classified: 81 out of 109 instances. This is an improvement from 77 previously.
Misclassified: Only 1 was misclassified as Coccidiosis (a notable decrease from 7), 17 as New Castle Disease (decreased from 20), and 10 as Salmonella (slight increase from 9). The model is now better at differentiating Healthy from Coccidiosis and New Castle Disease.
New Castle Disease (True Label, Row 2):

Correctly classified: 105 out of 113 instances. This is a very significant improvement from 76 previously!
Misclassified: 8 were misclassified as Healthy (slight decrease from 9), and 0 as Salmonella (decreased from 3). The model has become much more robust in identifying New Castle Disease.
Salmonella (True Label, Row 3):

Correctly classified: 107 out of 132 instances. This is a slight decrease from 115 previously.
Misclassified: It misclassified 3 as Coccidiosis (slight increase from 2), 9 as Healthy (slight increase from 7), and 13 as New Castle Disease (an increase from 8). The confusion with New Castle Disease has increased for Salmonella.
Overall Observations (After Class Weights):

Significant Improvement: The model shows clear improvement in classifying New Castle Disease and Healthy cases, with fewer misclassifications into other categories, especially into Coccidiosis.
Moderate Improvement: Coccidiosis classification also improved slightly, primarily by reducing its confusion with New Castle Disease.
Slight Degradation: The performance on Salmonella has slightly declined, with an increased tendency to confuse it with New Castle Disease and Healthy cases.
Persistent Confusion: While improved, some confusion still exists between Healthy and New Castle Disease, and now there's an increased confusion between Salmonella and New Castle Disease.
This re-evaluation after applying class weights shows that while some classes benefited significantly, leading to a higher overall accuracy (as seen in the test accuracy increase from ~82% to ~85%), the model's ability to distinguish Salmonella from other diseases, particularly New Castle Disease, became slightly more challenging.

Saving the Model

In [ ]:
model.save("kukusmart_model.h5")

In [ ]:
from tensorflow.keras.models import load_model

loaded_model = load_model("kukusmart_model.h5")
loaded_model.summary()

In [ ]:
print(train_generator.class_indices)

In [ ]:
class_labels = {v: k for k, v in train_generator.class_indices.items()}
print(class_labels)

In [ ]:
#saving model in a file
import json

with open("class_labels.json", "w") as f:
    json.dump(class_labels, f)

print("Class labels saved!")

In [ ]:
from google.colab import files

files.download("kukusmart_model.h5")
files.download("class_labels.json")